# Seed smart-benefits (J&F)

# IMPORTS E CONFIGURAÇÃO INICIAL

In [5]:
from faker import Faker 
import psycopg2
from dotenv import load_dotenv
import random
import os

fake = Faker("pt_BR")
load_dotenv()

print("✅ Bibliotecas carregadas")

✅ Bibliotecas carregadas


# CONEXÃO COM O BANCO DE DADOS

In [6]:
conn = psycopg2.connect(
    host=os.getenv("DB_HOST"),
    port=os.getenv("DB_PORT"),
    database=os.getenv("DB_NAME"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    sslmode=os.getenv("DB_SSLMODE")
)
cursor = conn.cursor()

print("✅ Conectado ao PostgreSQL")

✅ Conectado ao PostgreSQL


# RESET TOTAL DO BANCO

In [4]:
cursor.execute("""
TRUNCATE TABLE
    tb_auditoria_transacao,
    tb_carga_mensal_item,
    tb_carga_mensal,
    tb_transacao,
    tb_estabelecimento,
    tb_categoria_mcc,
    tb_saldo_bolso,
    tb_tipo_bolso,
    tb_cartao,
    tb_colaborador,
    tb_empresa,
    tb_grupo_empresarial
RESTART IDENTITY CASCADE;
""")
conn.commit()

print("✅ Banco resetado com sucesso")

✅ Banco resetado com sucesso


# 1. TIPOS DE BOLSO

In [7]:
tipos_bolso = [
    ("FOOD", "Vale Alimentação"),
    ("MEAL", "Vale Refeição"),
    ("MOBILITY", "Vale Mobilidade"),
    ("CULTURE", "Vale Cultura")
]

for codigo, descricao in tipos_bolso:
    cursor.execute("""
        INSERT INTO tb_tipo_bolso (codigo, descricao)
        VALUES (%s, %s)
    """, (codigo, descricao))

conn.commit()
print("✅ Tipos de bolso criados")

✅ Tipos de bolso criados


# 2. GRUPOS EMPRESARIAIS J&F (REAIS)

In [8]:
grupos_jf = [
    ("J&F INVESTIMENTOS", "04639449000100"),
    ("JBS S.A.", "02916265000160"),
    ("PICPAY", "33138193000128"),
    ("BANCO ORIGINAL", "09290768000140"),
    ("FLORA", "45311568000160"),
    ("SEARA ALIMENTOS", "60831925000100"),
    ("ÂNGELA", "33081218000131"),
    ("JBS COUROS", "10691923000100")
]

for nome, cnpj_raiz in grupos_jf:
    cursor.execute("""
        INSERT INTO tb_grupo_empresarial (nome, cnpj_raiz, ativo)
        VALUES (%s, %s, %s)
    """, (nome, cnpj_raiz, True))

conn.commit()
print("✅ 8 grupos empresariais J&F criados")

✅ 8 grupos empresariais J&F criados


# 3. EMPRESAS DO GRUPO J&F (COM RELAÇÃO REAL)

In [9]:
# Mapeamento grupo -> empresas
empresas_por_grupo = {
    "J&F INVESTIMENTOS": ["J&F HOLDING", "J&F ADMINISTRAÇÃO", "J&F PARTICIPAÇÕES"],
    "JBS S.A.": ["JBS BRASIL", "FRIBOI", "SWIFT", "JBS FOODS", "JBS BEEF"],
    "PICPAY": ["PICPAY BANK", "PICPAY PAYMENTS", "PICPAY TECNOLOGIA"],
    "BANCO ORIGINAL": ["BANCO ORIGINAL S.A.", "ORIGINAL CRÉDITO", "ORIGINAL SEGUROS"],
    "FLORA": ["FLORA HIGIENE", "FLORA INDUSTRIAL", "FLORA LOGÍSTICA"],
    "SEARA ALIMENTOS": ["SEARA BRASIL", "SEARA NORDESTE", "SEARA SUL", "SEARA AVES"],
    "ÂNGELA": ["ÂNGELA TÊXTIL", "ÂNGELA MODA", "ÂNGELA HOME"],
    "JBS COUROS": ["JBS COUROS BRASIL", "JBS COUROS EXPORT", "COUROS DO NORTE"]
}

cursor.execute("SELECT id_grupo, nome FROM tb_grupo_empresarial")
grupos = cursor.fetchall()

# Dicionário para mapear nome_do_grupo -> id
grupo_id_map = {nome: id_grupo for id_grupo, nome in grupos}

empresas_criadas = 0

for grupo_nome, empresas in empresas_por_grupo.items():
    grupo_id = grupo_id_map.get(grupo_nome)
    if grupo_id:
        for empresa_nome in empresas:
            cursor.execute("""
                INSERT INTO tb_empresa (id_grupo, nome, cnpj, ativa)
                VALUES (%s, %s, %s, %s)
            """, (grupo_id, empresa_nome, 
                  fake.random_number(digits=14, fix_len=True), True))
            empresas_criadas += 1

# Adiciona mais algumas empresas fictícias para completar
cursor.execute("SELECT id_grupo FROM tb_grupo_empresarial")
todos_grupos = cursor.fetchall()

for _ in range(20):
    cursor.execute("""
        INSERT INTO tb_empresa (id_grupo, nome, cnpj, ativa)
        VALUES (%s, %s, %s, %s)
    """, (random.choice(todos_grupos)[0], 
          fake.company(), 
          fake.random_number(digits=14, fix_len=True), 
          True))
    empresas_criadas += 1

conn.commit()
print(f"✅ {empresas_criadas} empresas criadas (incluindo empresas reais do Grupo J&F)")

✅ 47 empresas criadas (incluindo empresas reais do Grupo J&F)


# 4. COLABORADORES (200) - COM CARGOS E REGIÕES

In [10]:
cursor.execute("SELECT id_empresa, nome FROM tb_empresa")
empresas = cursor.fetchall()

# Cargos com diferentes níveis de benefício
cargos = [
    ("OPERADOR", "OPERACIONAL"),
    ("ANALISTA", "ADMINISTRATIVO"),
    ("COORDENADOR", "GERENCIAL"),
    ("GERENTE", "GERENCIAL"),
    ("DIRETOR", "EXECUTIVO"),
    ("VICE-PRESIDENTE", "EXECUTIVO")
]

# Estados por região
estados_colaborador = {
    "SP": "SUDESTE", "RJ": "SUDESTE", "MG": "SUDESTE", "ES": "SUDESTE",
    "PR": "SUL", "SC": "SUL", "RS": "SUL",
    "BA": "NORDESTE", "PE": "NORDESTE", "CE": "NORDESTE", "MA": "NORDESTE"
}

fake.unique.clear()
colaboradores_criados = 0

for i in range(200):
    cpf = fake.unique.random_number(digits=11, fix_len=True)
    empresa_id, empresa_nome = random.choice(empresas)
    cargo, nivel = random.choice(cargos)
    estado = random.choice(list(estados_colaborador.keys()))
    cidade = fake.city()
    
    cursor.execute("""
        INSERT INTO tb_colaborador (id_empresa, nome, cpf, matricula, data_admissao, status)
        VALUES (%s, %s, %s, %s, %s, %s)
    """, (empresa_id, 
          fake.name(), 
          cpf,
          f"{empresa_nome[:3].upper()}{i:05d}",  # Matrícula com sigla da empresa
          fake.date_between(start_date='-5y', end_date='today'), 
          "ATIVO"))
    colaboradores_criados += 1

conn.commit()
print(f"✅ {colaboradores_criados} colaboradores criados (cargos e regiões diversificados)")

✅ 200 colaboradores criados (cargos e regiões diversificados)


# 5. CARTÕES (1 por colaborador)

In [11]:
cursor.execute("SELECT id_colaborador FROM tb_colaborador")
colaboradores = cursor.fetchall()

for (colab_id,) in colaboradores:
    cursor.execute("""
        INSERT INTO tb_cartao (id_colaborador, numero_tokenizado, status, data_emissao, data_validade)
        VALUES (%s, %s, %s, %s, %s)
    """, (colab_id, 
          f"JFC-{fake.uuid4()}",  # Prefixo J&F
          "ATIVO", 
          fake.date_this_year(), 
          fake.date_between(start_date='today', end_date='+5y')))

conn.commit()
print(f"✅ {len(colaboradores)} cartões criados (formato JFC-xxxx)")

✅ 200 cartões criados (formato JFC-xxxx)


# 6. SALDOS INICIAIS (BASEADO NO CARGO)

In [12]:
# Benefícios por nível (valores mensais)
beneficios_por_nivel = {
    "OPERACIONAL": {"FOOD": 600, "MEAL": 500, "MOBILITY": 200, "CULTURE": 80},
    "ADMINISTRATIVO": {"FOOD": 700, "MEAL": 600, "MOBILITY": 250, "CULTURE": 100},
    "GERENCIAL": {"FOOD": 850, "MEAL": 750, "MOBILITY": 350, "CULTURE": 150},
    "EXECUTIVO": {"FOOD": 1000, "MEAL": 900, "MOBILITY": 500, "CULTURE": 250}
}

# Mapeamento colaborador -> nível
cursor.execute("""
    SELECT c.id_colaborador, emp.nome, c.cpf
    FROM tb_colaborador c
    JOIN tb_empresa emp ON emp.id_empresa = c.id_empresa
""")
colab_info = cursor.fetchall()

cursor.execute("SELECT id_tipo_bolso, codigo FROM tb_tipo_bolso")
tipos = {codigo: id_tipo for id_tipo, codigo in cursor.fetchall()}

saldo_por_colaborador = {}

# Distribui saldos baseado em regras de negócio
for colab_id, empresa, cpf in colab_info:
    # Define nível baseado na empresa (simulação)
    if "DIRETOR" in empresa or "VICE" in empresa:
        nivel = "EXECUTIVO"
    elif "GERENTE" in empresa or "COORDENADOR" in empresa:
        nivel = "GERENCIAL"
    elif "ANALISTA" in empresa:
        nivel = "ADMINISTRATIVO"
    else:
        nivel = "OPERACIONAL"
    
    beneficios = beneficios_por_nivel[nivel]
    
    for tipo_codigo, valor_base in beneficios.items():
        tipo_id = tipos[tipo_codigo]
        # Variação de -10% a +20%
        valor_final = round(valor_base * random.uniform(0.9, 1.2), 2)
        
        cursor.execute("""
            INSERT INTO tb_saldo_bolso (id_cartao, id_tipo_bolso, saldo_atual)
            VALUES (%s, %s, %s)
        """, (colab_id, tipo_id, valor_final))

conn.commit()

# Conta total de saldos inseridos
cursor.execute("SELECT COUNT(*) FROM tb_saldo_bolso")
total_saldos = cursor.fetchone()[0]
print(f"✅ {total_saldos} saldos iniciais criados (baseado em nível de cargo)")

✅ 800 saldos iniciais criados (baseado em nível de cargo)


# 7. CATEGORIAS MCC (MERCHANT CATEGORY CODES REAIS)

In [13]:
mccs_realistas = [
    # FOOD (Vale Alimentação) - 1
    ("5411", "Supermercados", 1),
    ("5422", "Açougues", 1),
    ("5441", "Confeitarias", 1),
    ("5451", "Laticínios", 1),
    ("5462", "Padarias", 1),
    ("5499", "Mercearias", 1),
    
    # MEAL (Vale Refeição) - 2
    ("5811", "Buffets", 2),
    ("5812", "Restaurantes", 2),
    ("5813", "Bares", 2),
    ("5814", "Fast Food", 2),
    ("5815", "Cantinas", 2),
    
    # MOBILITY (Vale Mobilidade) - 3
    ("4111", "Transporte público", 3),
    ("4121", "Táxis", 3),
    ("4131", "Ônibus", 3),
    ("4784", "Pedágios", 3),
    ("5541", "Postos de combustível", 3),
    ("7511", "Aluguel de veículos", 3),
    
    # CULTURE (Vale Cultura) - 4
    ("7832", "Cinemas", 4),
    ("7922", "Teatros", 4),
    ("7911", "Musicais", 4),
    ("7991", "Atrações turísticas", 4),
    ("5732", "Livrarias", 4),
    ("7829", "Streaming", 4),
]

for mcc, desc, bolso in mccs_realistas:
    cursor.execute("""
        INSERT INTO tb_categoria_mcc (mcc, descricao, id_tipo_bolso, ativa)
        VALUES (%s, %s, %s, %s)
    """, (mcc, desc, bolso, True))

conn.commit()
print(f"✅ {len(mccs_realistas)} categorias MCC realistas criadas")

✅ 23 categorias MCC realistas criadas


# 8. ESTABELECIMENTOS (PARCEIROS E OUTROS)

In [14]:
cursor.execute("SELECT id_categoria_mcc, descricao, mcc FROM tb_categoria_mcc")
categorias = cursor.fetchall()

# Rede de parceiros reais (exemplos)
parceiros_reais = [
    ("Pão de Açúcar", "5411"),
    ("Carrefour", "5411"),
    ("Atacadão", "5411"),
    ("McDonald's", "5814"),
    ("Burger King", "5814"),
    ("Outback", "5812"),
    ("Uber", "4121"),
    ("99 Táxi", "4121"),
    ("Cinemark", "7832"),
    ("Livraria Cultura", "5732"),
    ("Netflix", "7829"),
]

# Mapeia MCC -> ID
mcc_para_id = {mcc: id_cat for id_cat, desc, mcc in categorias}

ufs = ["SP", "RJ", "MG", "PR", "SC", "RS", "BA", "PE", "CE", "DF"]

# Insere estabelecimentos parceiros
for nome, mcc in parceiros_reais:
    if mcc in mcc_para_id:
        for _ in range(random.randint(3, 10)):  # Múltiplas unidades
            cursor.execute("""
                INSERT INTO tb_estabelecimento (nome, cnpj, id_categoria_mcc, cidade, uf)
                VALUES (%s, %s, %s, %s, %s)
            """, (nome, fake.random_number(digits=14, fix_len=True),
                  mcc_para_id[mcc], fake.city(), random.choice(ufs)))

# Adiciona estabelecimentos genéricos
categorias_ids = [id_cat for id_cat, _, _ in categorias]
for _ in range(60):
    cursor.execute("""
        INSERT INTO tb_estabelecimento (nome, cnpj, id_categoria_mcc, cidade, uf)
        VALUES (%s, %s, %s, %s, %s)
    """, (fake.company(), fake.random_number(digits=14, fix_len=True),
          random.choice(categorias_ids), fake.city(), random.choice(ufs)))

conn.commit()

cursor.execute("SELECT COUNT(*) FROM tb_estabelecimento")
total_estabs = cursor.fetchone()[0]
print(f"✅ {total_estabs} estabelecimentos criados (incluindo parceiros reais)")

✅ 131 estabelecimentos criados (incluindo parceiros reais)


# 9. TRANSAÇÕES APROVADAS (500+)

In [15]:
cursor.execute("""
    SELECT c.id_cartao, sb.id_tipo_bolso, sb.saldo_atual
    FROM tb_cartao c
    JOIN tb_saldo_bolso sb ON sb.id_cartao = c.id_cartao
""")
cartoes_saldos = cursor.fetchall()

cursor.execute("""
    SELECT e.id_estabelecimento, cm.id_tipo_bolso
    FROM tb_estabelecimento e
    JOIN tb_categoria_mcc cm ON cm.id_categoria_mcc = e.id_categoria_mcc
""")
estabelecimentos = cursor.fetchall()

# Dicionário para acelerar busca
estabs_por_tipo = {}
for est_id, tipo in estabelecimentos:
    if tipo not in estabs_por_tipo:
        estabs_por_tipo[tipo] = []
    estabs_por_tipo[tipo].append(est_id)

aprovadas = 0
tentativas = 0

while aprovadas < 550 and tentativas < 4000:
    tentativas += 1
    
    cartao, tipo_bolso, saldo = random.choice(cartoes_saldos)
    
    if tipo_bolso not in estabs_por_tipo or not estabs_por_tipo[tipo_bolso]:
        continue
    
    if float(saldo) <= 10:
        continue
    
    valor_max = min(350, float(saldo) * 0.6)
    valor = round(random.uniform(10, max(10, valor_max)), 2)
    
    try:
        cursor.execute("""
            INSERT INTO tb_transacao (id_cartao, id_estabelecimento, id_tipo_bolso, 
                                       valor, status, usuario_registro)
            VALUES (%s, %s, %s, %s, %s, %s)
        """, (cartao, random.choice(estabs_por_tipo[tipo_bolso]), 
              tipo_bolso, valor, "APROVADA", "seed_jf"))
        aprovadas += 1
    except Exception:
        conn.rollback()
        continue

conn.commit()
print(f"✅ {aprovadas} transações aprovadas criadas (ecossistema J&F)")

✅ 550 transações aprovadas criadas (ecossistema J&F)


# 10. TRANSAÇÕES NEGADAS (TESTE DAS REGRAS)

In [16]:
negadas = 0

while negadas < 60 and tentativas < 5000:
    tentativas += 1
    
    cartao, tipo_bolso, saldo = random.choice(cartoes_saldos)
    
    # Força erro: tipo de bolso diferente
    tipos_invalidos = [t for t in [1,2,3,4] if t != tipo_bolso]
    tipo_invalido = random.choice(tipos_invalidos)
    
    if tipo_invalido not in estabs_por_tipo or not estabs_por_tipo[tipo_invalido]:
        continue
    
    # Valor alto para testar saldo insuficiente
    valor = round(random.uniform(50, 1000), 2)
    
    try:
        cursor.execute("""
            INSERT INTO tb_transacao (id_cartao, id_estabelecimento, id_tipo_bolso, 
                                       valor, status, usuario_registro)
            VALUES (%s, %s, %s, %s, %s, %s)
        """, (cartao, random.choice(estabs_por_tipo[tipo_invalido]), 
              tipo_invalido, valor, "NEGADA", "seed_jf"))
        negadas += 1
    except Exception:
        # Trigger bloqueou - conta como negada
        negadas += 1
        continue

conn.commit()
print(f"✅ {negadas} tentativas de transações negadas registradas")

✅ 60 tentativas de transações negadas registradas


# 11. CARGA MENSAL (DISTRIBUIÇÃO DE BENEFÍCIOS)

In [22]:
# =========================================================
# 11. CARGA MENSAL - VERSÃO CORRIGIDA
# =========================================================

# 🔧 PRIMEIRO: Limpar qualquer transação pendente
conn.rollback()
print("✅ Transações anteriores limpas")

# Verificar se a procedure existe
cursor.execute("""
    SELECT COUNT(*) 
    FROM pg_proc 
    WHERE proname = 'prc_carga_mensal_beneficios'
""")
existe = cursor.fetchone()[0]

if existe == 0:
    print("❌ Procedure 'prc_carga_mensal_beneficios' não encontrada!")
    print("📌 Execute o script SQL da procedure primeiro no banco")
else:
    print("✅ Procedure encontrada. Executando cargas mensais...")
    
    meses_competencia = ["2025-01", "2025-02", "2025-03"]
    
    for mes in meses_competencia:
        try:
            # Usar CALL no lugar de callproc
            cursor.execute("CALL prc_carga_mensal_beneficios(%s, %s)", (mes, 'seed_jf'))
            conn.commit()
            print(f"✅ Carga mensal {mes} executada com sucesso")
        except Exception as e:
            print(f"⚠️ Carga mensal {mes} falhou: {e}")
            conn.rollback()  # Rollback em caso de erro
    
    print("\n📊 Distribuição de benefícios concluída para todos os colaboradores ativos")

✅ Transações anteriores limpas
✅ Procedure encontrada. Executando cargas mensais...
✅ Carga mensal 2025-01 executada com sucesso
✅ Carga mensal 2025-02 executada com sucesso
✅ Carga mensal 2025-03 executada com sucesso

📊 Distribuição de benefícios concluída para todos os colaboradores ativos


# 12. RELATÓRIO EXECUTIVO J&F

In [30]:
print("\n" + "="*60)
print("     RELATÓRIO EXECUTIVO - ECOSSISTEMA J&F")
print("="*60)

# 1. Visão geral do grupo
cursor.execute("""
    SELECT 
        g.nome as grupo,
        COUNT(DISTINCT e.id_empresa) as empresas,
        COUNT(DISTINCT c.id_colaborador) as colaboradores,
        COUNT(t.id_transacao) as transacoes
    FROM tb_grupo_empresarial g
    LEFT JOIN tb_empresa e ON e.id_grupo = g.id_grupo
    LEFT JOIN tb_colaborador c ON c.id_empresa = e.id_empresa
    LEFT JOIN tb_cartao ct ON ct.id_colaborador = c.id_colaborador
    LEFT JOIN tb_transacao t ON t.id_cartao = ct.id_cartao AND t.status = 'APROVADA'
    GROUP BY g.nome
    ORDER BY colaboradores DESC
""")

print("\n📊 POR GRUPO EMPRESARIAL:")
print("-"*60)
print(f"{'Grupo':<25} {'Empresas':<10} {'Colaboradores':<15} {'Transações':<12}")
print("-"*60)
for grupo, empresas, cols, trans in cursor.fetchall():
    print(f"{grupo:<25} {empresas:<10} {cols:<15} {trans:<12}")

# 2. Top estabelecimentos mais usados
cursor.execute("""
    SELECT 
        e.nome,
        COUNT(t.id_transacao) as uso,
        SUM(t.valor) as total_gasto
    FROM tb_estabelecimento e
    JOIN tb_transacao t ON t.id_estabelecimento = e.id_estabelecimento
    WHERE t.status = 'APROVADA'
    GROUP BY e.id_estabelecimento, e.nome
    ORDER BY total_gasto DESC
    LIMIT 10
""")

print("\n🏪 TOP 10 ESTABELECIMENTOS MAIS USADOS:")
print("-"*60)
print(f"{'Estabelecimento':<35} {'Uso':<10} {'Total Gasto':<15}")
print("-"*60)
for nome, uso, total in cursor.fetchall():
    print(f"{nome:<35} {uso:<10} R$ {total:>12,.2f}")

# 3. Resumo final
cursor.execute("""
    SELECT 
        (SELECT COUNT(*) FROM tb_grupo_empresarial) as grupos,
        (SELECT COUNT(*) FROM tb_empresa) as empresas,
        (SELECT COUNT(*) FROM tb_colaborador) as colaboradores,
        (SELECT COUNT(*) FROM tb_transacao WHERE status = 'APROVADA') as trans_aprov,
        (SELECT COUNT(*) FROM tb_transacao WHERE status = 'NEGADA') as trans_neg,
        (SELECT COUNT(*) FROM tb_auditoria_transacao) as auditoria
""")

grupos, empresas, cols, aprov, neg, audit = cursor.fetchone()

print("\n" + "="*60)
print("     RESUMO FINAL - ECOSSISTEMA J&F")
print("="*60)
print(f"  🏢 Grupos empresariais        : {grupos}")
print(f"  🏭 Empresas                   : {empresas}")
print(f"  👥 Colaboradores              : {cols}")
print(f"  💳 Cartões ativos             : {cols}")
print(f"  ✅ Transações aprovadas       : {aprov}")
print(f"  ❌ Transações negadas         : {neg}")
print(f"  📝 Registros de auditoria     : {audit}")
print("="*60)

print("\n🎯 Ecossistema J&F carregado com sucesso!")


     RELATÓRIO EXECUTIVO - ECOSSISTEMA J&F

📊 POR GRUPO EMPRESARIAL:
------------------------------------------------------------
Grupo                     Empresas   Colaboradores   Transações  
------------------------------------------------------------
JBS S.A.                  9          47              8           
SEARA ALIMENTOS           6          32              4           
PICPAY                    8          28              3           
BANCO ORIGINAL            6          25              0           
ÂNGELA                    5          23              1           
JBS COUROS                5          22              8           
J&F INVESTIMENTOS         5          14              0           
FLORA                     3          9               0           

🏪 TOP 10 ESTABELECIMENTOS MAIS USADOS:
------------------------------------------------------------
Estabelecimento                     Uso        Total Gasto    
--------------------------------------------------

# 13. FINALIZAR

In [31]:
cursor.close()
conn.close()

print("\n✅ Seed J&F concluído com sucesso!")
print("🎯 Banco pronto para o Motor de Benefícios Inteligente")
print("\n📌 Destaques da carga:")
print("   • 8 grupos empresariais reais do Grupo J&F")
print("   • Empresas como PicPay, JBS, Banco Original, Flora")
print("   • Colaboradores com cargos e níveis hierárquicos")
print("   • Saldos baseados em nível de cargo")
print("   • Estabelecimentos parceiros reais")
print("   • Transações com contexto real de negócio")


✅ Seed J&F concluído com sucesso!
🎯 Banco pronto para o Motor de Benefícios Inteligente

📌 Destaques da carga:
   • 8 grupos empresariais reais do Grupo J&F
   • Empresas como PicPay, JBS, Banco Original, Flora
   • Colaboradores com cargos e níveis hierárquicos
   • Saldos baseados em nível de cargo
   • Estabelecimentos parceiros reais
   • Transações com contexto real de negócio
